# 01 — Moving Average

**Reference:** Vandeput, N. (2021). *Data Science for Supply Chain Forecasting* (2nd ed.), Chapter 1.

The moving average is the simplest forecasting model — and the right place to start.
It assumes future demand is similar to recent demand, averaging the last `n` periods.

$$f_t = \frac{1}{n} \sum_{i=1}^{n} d_{t-i}$$

**Limitations to keep in mind:**
- Cannot capture trend
- Cannot capture seasonality
- Weights all historical periods equally


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. The Model

In [ ]:
def moving_average(d, extra_periods=12, n=3):
    """
    Moving average forecast.

    Parameters
    ----------
    d : array-like — historical demand
    extra_periods : int — number of future periods to forecast
    n : int — number of periods to average

    Returns
    -------
    pd.DataFrame with columns: Demand, Forecast, Error
    """
    cols = len(d)
    d = np.append(d, [np.nan] * extra_periods)
    f = np.full(cols + extra_periods, np.nan)

    for t in range(n, cols):
        f[t] = np.mean(d[t - n:t])

    # All future forecasts = last computed average
    f[t + 1:] = np.mean(d[t - n + 1:t + 1])

    return pd.DataFrame({'Demand': d, 'Forecast': f, 'Error': d - f})

## 2. Simulated FMCG Demand

Simulating 36 months of beverage demand — flat trend, moderate noise.

In [ ]:
np.random.seed(42)
n_periods = 36
base_demand = 1000
noise = np.random.normal(0, 80, n_periods)
demand = base_demand + noise
demand = np.clip(demand, 0, None)  # no negative demand

print(f'Mean demand: {demand.mean():.0f} units')
print(f'Std deviation: {demand.std():.0f} units')

## 3. Compare n=3 vs n=8

In [ ]:
df3 = moving_average(demand, extra_periods=6, n=3)
df8 = moving_average(demand, extra_periods=6, n=8)

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(df3['Demand'], label='Actual Demand', color='#2C2C2A', linewidth=1.5)
ax.plot(df3['Forecast'], label='MA(3) — reactive', color='#185FA5',
        linestyle='--', linewidth=1.5)
ax.plot(df8['Forecast'], label='MA(8) — smooth', color='#D85A30',
        linestyle='--', linewidth=1.5)

ax.axvline(x=len(demand), color='gray', linestyle=':', alpha=0.6, label='Forecast horizon')
ax.set_title('Moving Average: Reactivity vs Smoothness Trade-off', fontsize=13)
ax.set_xlabel('Period (months)')
ax.set_ylabel('Demand (units)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Forecast KPIs

We'll build a full KPI module in notebook `06_forecast_kpis.ipynb`, but let's compute MAE here as a quick check.

In [ ]:
def mae(df):
    """Mean Absolute Error — average absolute forecast error."""
    errors = df['Error'].dropna()
    return np.mean(np.abs(errors))

def bias(df):
    """Bias — average signed error. Positive = under-forecasting."""
    return df['Error'].dropna().mean()

print('--- MA(3) ---')
print(f'MAE:  {mae(df3):.1f} units')
print(f'Bias: {bias(df3):.1f} units')
print()
print('--- MA(8) ---')
print(f'MAE:  {mae(df8):.1f} units')
print(f'Bias: {bias(df8):.1f} units')

## 5. Key Takeaways

| n | Behaviour | Use when |
|---|-----------|----------|
| Small (1-3) | Reacts fast, sensitive to noise | Demand changes frequently |
| Large (6-12) | Smooth, slow to react | Stable demand, noise reduction needed |

**Next:** Exponential smoothing (`02_exponential_smoothing.ipynb`) solves the flat-weighting
limitation by giving more weight to recent observations.
